[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/finetuning/blob/main/chapter_08/listing_8.1.ipynb)

In [1]:
import sys
if "google.colab" in sys.modules:
    !pip install -q -U transformers datasets peft


### Listing 6.30: Preparing Qwen2.5-0.5B and the Cleaned Alpaca Dataset

In [2]:
import torch
import math
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    dtype=torch.float16,
    device_map="auto"
)

model.eval()

dataset = load_dataset("yahma/alpaca-cleaned", split="train[:50]")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

### Listing 6.31: Functions for Computing the Perplexity



In [3]:
def format_sample(example):
    return f"###Question: {example['instruction']}\n###Answer: {example['output']}"

def calculate_sequence_ppl(text):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]
    
    if input_ids.size(1) <= 1:
        return float('inf')
        
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        neg_log_likelihood = outputs.loss
        perplexity = torch.exp(neg_log_likelihood).item()
        
    return perplexity

### Listing 6.32: Computing Perplexity for the Cleaned Alpaca Dataset

In [4]:
recorded_perplexity = []
for i, example in tqdm(enumerate(dataset)):
    formatted_text = format_sample(example)
    ppl = calculate_sequence_ppl(formatted_text)
    recorded_perplexity.append(ppl)

p_avg = sum(recorded_perplexity) / len(recorded_perplexity) 
variance = sum((x - p_avg) ** 2 for x in recorded_perplexity) / len(recorded_perplexity)
p_std = math.sqrt(variance)

print("--- Perplexity Statistics ---")
print(f"Count:   {len(recorded_perplexity)}")
print(f"Min:     {min(recorded_perplexity):.4f}")
print(f"Max:     {max(recorded_perplexity):.4f}")
print(f"Average: {p_avg:.4f}")
print(f"Std Dev: {p_std:.4f}")

50it [00:01, 39.16it/s]

--- Perplexity Statistics ---
Count:   50
Min:     2.6056
Max:     27.7277
Average: 7.6030
Std Dev: 6.2185


### Listing 6.33: How to Set up a LoRA Configuration and Calculate the Parameters Programmatically

In [5]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_id = "google/gemma-3-1b-it"
base_model = AutoModelForCausalLM.from_pretrained(model_id)

lora_config = LoraConfig(
    r=16,  
    lora_alpha=32,  
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

lora_model = get_peft_model(base_model, lora_config)

lora_model.print_trainable_parameters()

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

trainable params: 1,490,944 || all params: 1,001,376,896 || trainable%: 0.1489


### Listing 6.34: How to Count the Tokens of a Dataset

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
dataset = load_dataset("HuggingFaceH4/no_robots", split="train")

def calculate_token_length(example):
    try:
        text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    except Exception:
        text = " ".join([m["content"] for m in example["messages"]])
        
    return {"token_count": len(tokenizer(text)["input_ids"])}

tokenized_dataset = dataset.map(calculate_token_length, num_proc=4)
total_examples = len(dataset)
total_tokens = sum(tokenized_dataset["token_count"])

print("-" * 30)
print(f"Total Examples: {total_examples:,}")
print(f"Total Tokens:   {total_tokens:,}")

------------------------------
Total Examples: 9,500
Total Tokens:   2,798,810
